In [0]:
# step-6 Write a function which can do SCD-Tpye1 if we provide parameters like dataframe,buisness key and target table | After creating these function (scd-1) we will take it to common_utils and then reuse it here and in other notebooks
# step-7 Fully modularize the code (hardcoded values to json config, proper logging setup and all the resusable function should have used)

### Read customers table from bronze and store in df

In [0]:
import pyspark.sql.functions as F
df=spark.read.table("retaildataplatform.bronze.sqlserver_customers")
df.display()

### step-2 Display the dataframe and try to identify what transformations can be done

In [0]:
import pyspark.sql.functions as F
# 1. Customer name should be small
df=df.withColumn('customer_name',F.lower(df['customer_name']))
# customer_name first name and last name drop customer_name
df=df.withColumn('name_parts',F.split(F.col('customer_name'),","))\
    .withColumn('Firs_name',F.expr("get(name_parts,1)"))\
    .withColumn('Last_name',F.expr("get(name_parts,0)"))\
    .drop('name_parts')\
    .drop('customer_name')
# 2. Postcode should not contain decimal value
df=df.withColumn('postcode', F.regexp_replace(F.col('postcode'), r'\.0$', ''))
# 3. City should be capital
df=df.withColumn('city',F.upper(F.col('city')))
# 4. Drop file path
df=df.drop('file_path')
# 5. Valid from and Valid to should utc timestamp
df=df.withColumn('valid_from',F.from_unixtime('valid_from').cast('timestamp'))
df=df.withColumn('valid_to',F.from_unixtime(F.expr("try_cast(valid_to as long)")).cast('timestamp'))
# 6. Add Country column with value USA
df = df.withColumn('country',F.lit('USA'))

### step-3 Apply the required transformation steps which you identified and then again store in dataframe

In [0]:
df.show()

In [0]:

df.display()

df=df.dropDuplicates(['customer_id'])
df.groupBy('customer_id').count().filter('count>1').display()

### step-4 Write a function in same notebook which clean and transform the customers data from bronze and returns the dataframe

In [0]:
import pyspark.sql.functions as F
df=spark.read.table("retaildataplatform.bronze.sqlserver_customers")
def transform_column(df):
    df=df.withColumn('customer_name',F.lower(df['customer_name']))\
        .withColumn('name_parts',F.split(F.col('customer_name'),","))\
        .withColumn('Firs_name',F.expr("get(name_parts,1)"))\
        .withColumn('Last_name',F.expr("get(name_parts,0)"))\
        .withColumn('postcode', F.regexp_replace(F.col('postcode'), r'\.0$', ''))\
        .withColumn('city',F.upper(F.col('city')))\
        .withColumn('valid_from',F.from_unixtime('valid_from').cast('timestamp'))\
        .withColumn('valid_to',F.from_unixtime(F.expr("try_cast(valid_to as long)")).cast('timestamp'))\
        .withColumn('country',F.lit('USA'))

    df=df.drop('name_parts','customer_name','file_path')
    df=df.dropDuplicates(['customer_id'])

    return df.select(
    'customer_id',
    'Firs_name',
    'Last_name',
    'tax_id',
    'tax_code',
    'state',
    'city',
    'country',
    'postcode',
    'street',
    'number',
    'unit',
    'region',
    'district',
    'lon',
    'lat',
    'ship_to_address',
    'valid_from',
    'valid_to',
    'units_purchased',
    'loyalty_segment',
    'last_updated_ts',
    )

In [0]:
df=spark.read.table("retaildataplatform.bronze.sqlserver_customers")
cleaned_customer=transform_column(df)
cleaned_customer.display()

### step-5 In Final dataframe we will apply SCD-Type1 and then store in silver schema(before that we need to create silver schema using SQL)

### SCD-1
Upsert>>update+insert
Update-> Changed Rows
Insert-> New Rows

Buisness key or Unique Identifier
SCD-1 ->> Compare Source and target

no silver schema
no customer table in silver

check if table exist or not
if exist then do scd-1

otherwise do full load (write the table in delta format)

In [0]:
if spark.catalog.tableExists("retaildataplatform.silver.customers"):
    print('Table Exists proceed with SCD-1 process.')
else:
    print('Table does not exist- First create Schema(Table)')
    spark.sql("create schema if not exists retaildataplatform.silver")
    df.write.format('delta').saveAsTable('retaildataplatform.silver.customers')